# Homogeneous correlation clustering (cluster shrinkage)
**ARPM script:** `s_shrink_corr_hom_clusters` · **Lab:** Correlation clustering

**In plain words:** a 500×500 correlation matrix has ~125k numbers, most of them noise.
If we trust the clusters, replace every correlation **inside** a cluster by that cluster's average,
and every correlation **between** two clusters by their cross-average. Few parameters, much less noise: a shrinkage estimator.
If the result isn't a valid correlation matrix (an eigenvalue ≤ 0), shrink the off-diagonals towards 0 by $\gamma$ until it is.

Example: 5 stocks, clusters {0,1} and {2,3,4} (`l_clus = [0, 2, 5]`).

Changes from ARPM's original: data path via `clustering_lab.paths`. Code otherwise unchanged.

> **Watch the inner loop:** `for k2 in range(k1, k_bar)` starts at `k1`, so the within-block value is overwritten by the
> plain block average *including the 1s on the diagonal*. The cross-check cell at the bottom shows the difference vs the
> off-diagonal average that the first line of the loop intends.

In [ ]:
# --- PyCharm plumbing (not ARPM code) -------------------------------------
# Find the unit root from wherever the notebook runs, so `clustering_lab`
# imports and the data path work in PyCharm, Jupyter or a test run.
import sys
from pathlib import Path

UNIT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "clustering_lab").is_dir())
if str(UNIT_ROOT) not in sys.path:
    sys.path.insert(0, str(UNIT_ROOT))

from clustering_lab.paths import stocks_csv_path, OUTPUTS_DIR
STOCKS_CSV = stocks_csv_path()   # data/db_stocks_sp.csv (or $ARPM_DB_STOCKS_SP)
print("data:", STOCKS_CSV)

In [ ]:
from numpy import argsort, array, average, corrcoef, diag, diagflat, \
                  diff, eye, ix_, log, ones, sum, zeros
from numpy.linalg import det, eigvalsh
from pandas import read_csv
from seaborn import heatmap
from matplotlib.pyplot import show, xticks, yticks

## Load data

In [ ]:
db_stocks_sp = read_csv(STOCKS_CSV, index_col=0, header=[0, 1])
v = array(db_stocks_sp)  # stocks values
n_bar = v.shape[1]  # number of stocks
sectors = array(db_stocks_sp.columns.levels[0])  # sectors names
labels = array(db_stocks_sp.columns.codes)[0, :]  # sectors indices

## 1. Compute correlation matrix of log-returns

In [ ]:
eps = diff(log(v), axis=0)  # log-returns
c2_hist = corrcoef(eps.T)  # historical correlation
i_s = argsort(labels)  # sorting index for sectors
c2_sec = c2_hist[ix_(i_s, i_s)]  # correlation matrix sorted by sectors

## 2. Perform homogeneous correlation cluster shrinkage

In [ ]:
index = [96, 97, 128, 132, 138]  # entries of interest (you can change it)
c2_bar = c2_sec[ix_(index, index)]  # correlation

# compute optimal clustering for matrix c2_bar
i_clus = range(len(index))
c2_c = c2_bar[ix_(i_clus, i_clus)]

# homogenize correlation using views
l_clus = [0, 2, 5]  # vector such that l_hom(k+1):l_hom[k], vector of entries in k-th cluster (you can change it)
k_bar = len(l_clus) - 1
l_diff = diff(l_clus)
c2_clus = zeros((len(i_clus), len(i_clus)))
threshold = 10**-5  # threshold for determinant equal to zero (you can change it)
gamma = 1  # shrinkage coefficient to ensure positivity (you can change it)
step = 0.005  # routine step (you can change it)
for k1 in range(k_bar):
    d = range(l_clus[k1], l_clus[k1 + 1])
    b = l_diff[k1]
    if b > 1:
        c2_tmp = sum(c2_c[ix_(d, d)] - eye(b))/(b*(b - 1))
        c2_clus[ix_(d, d)] = c2_tmp*ones((b, b))
    for k2 in range(k1, k_bar):
        d1 = range(l_clus[k2], l_clus[k2 + 1])
        c2_tmp = average(c2_c[ix_(d, d1)])
        b1 = l_diff[k2]
        c2_clus[ix_(d, d1)] = c2_tmp*ones((b, b1))
        c2_clus[ix_(d1, d)] = c2_tmp*ones((b1, b))
        # 1 on main diagonals
        c2_clus = c2_clus - diagflat(diag(c2_clus)) + diagflat(ones((1, len(i_clus))))
        c2_hom = c2_clus


# ensure positivity
lam_min = min(eigvalsh(c2_hom)) # smaller eigenvalues computation
while lam_min <= threshold:
        gamma = gamma - step
        # homogenized correlation matrix
        c2_hom = gamma*c2_hom
        # 1 on main diagonals
        c2_hom = c2_hom - diagflat(diag(c2_hom)) + diagflat(ones((1, len(i_clus))))
        # smallest eigenvalue update
        lam_min = min(eigvalsh(c2_hom))
        if gamma <= 0:
            print('convergence not reached')

############################################### plots ###############################################
use_symbols = True  # True to use stocks symbols as labels, False to use sectors (you can change it)
if use_symbols:
    ticks_labels = array([k for j,k in db_stocks_sp.columns.values])
else:
    ticks_labels = array([j for j,k in db_stocks_sp.columns.values])

# starting correlation
heatmap(c2_bar, square=True, annot=True, fmt='.2f')
xticks(i_clus, ticks_labels[index], rotation=90); yticks(i_clus, ticks_labels[index]); show();

# homogenized correlation
heatmap(c2_hom, square=True, annot=True, fmt='.2f')
xticks(i_clus, ticks_labels[index], rotation=90); yticks(i_clus, ticks_labels[index]); show();

## Cross-check with `clustering_lab` (not ARPM code)
`homogenize_correlation` uses the **off-diagonal** average inside each block. Compare with ARPM's matrix above.
(Note also: ARPM's tick labels index the *unsorted* column list with `index`, while `c2_bar` comes from the *sector-sorted* matrix, so the tickers shown may not be the stocks plotted. `ticks_labels[i_s][index]` would be the matching names.)

In [ ]:
from clustering_lab.corr_cluster_tools import homogenize_correlation

c2_hom_lab, gamma_lab = homogenize_correlation(c2_bar, l_clus)
print("ARPM loop:\n", c2_hom.round(3))
print("\nclustering_lab (off-diagonal block averages), gamma =", round(gamma_lab, 3), ":\n", c2_hom_lab.round(3))
print("\nstocks actually plotted:", ticks_labels[i_s][index])